# Preparando os arquivos e dependências

In [ ]:
print("Instalando bibliotecas necessárias...")
%pip install -q ultralytics "timm>=0.6.13"
%pip install -q git+https://github.com/baudm/parseq.git
%pip install -q pytorch-lightning
print("Instalação concluída.")

import torch
import ultralytics

ultralytics.checks()
print(f"GPU disponível: {torch.cuda.is_available()}")

Puxando arquivo do drive (Dataset LRLPR-26)

In [ ]:
import os
import cv2
import matplotlib.pyplot as plt
import zipfile
from google.colab import drive

drive.mount('/content/drive')

#-------------------------- pegando zip do drive
if not os.path.exists('/content/base_competicao'):
  zip_path = "/content/drive/MyDrive/<CAMINHO_DATASET>/LRLPR-26-5opEvJTW.zip"   # caminho zip
  destino = "/content/base_competicao"

  # overwrite se ja tem pasta
  !unzip -oq {zip_path} -d {destino}

  # remover a pasta __MACOSX
  !rm -rf {destino}/__MACOSX

print("Dataset pronto em /content/base_competicao")
#----------------------------------------------------------------------------------

Verificando tracks e imagens de teste por categoria

In [ ]:
import os
import json
from collections import defaultdict

test_base = '/content/base_competicao/LRLPR-26-5opEvJTW/test'

imgs_por_categoria = defaultdict(int)
tracks_por_categoria = defaultdict(int)

for track_name in os.listdir(test_base):
    track_path = os.path.join(test_base, track_name)
    if not os.path.isdir(track_path):
        continue

    # categoria = layout da placa (ex: Brazilian, Mercosur)
    annotation_path = os.path.join(track_path, 'annotations.json')
    if os.path.exists(annotation_path):
        with open(annotation_path) as f:
            ann = json.load(f)
        categoria = ann.get('plate_layout', 'unknown')
    else:
        categoria = 'unknown'

    imgs = [f for f in os.listdir(track_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    imgs_por_categoria[categoria] += len(imgs)
    tracks_por_categoria[categoria] += 1

print("─── Imagens por categoria ───")
for cat in sorted(imgs_por_categoria):
    print(f"  {cat}: {imgs_por_categoria[cat]} imagens em {tracks_por_categoria[cat]} tracks")
print(f"\n  Total: {sum(imgs_por_categoria.values())} imagens em {sum(tracks_por_categoria.values())} tracks")

Funções de avaliação (MVCP e CER)

In [ ]:
import numpy as np
from collections import defaultdict, Counter

# Dicionário para guardar o resumo de todos os modelos avaliados na sessão
historico_modelos = {}

def mvcp(predictions):
    """
    predictions: lista de (texto, tensor_de_scores_por_char)
    Retorna a predição final da track via Majority Vote by Character Position.
    """
    if not predictions:
        return ""

    # limpa caracteres não alfanuméricos (ex: '-' em placas BR)
    clean = []
    for text, scores in predictions:
        text_clean = []
        scores_clean = []
        for char, score in zip(text, scores):
            if char.isalnum():
                text_clean.append(char)
                scores_clean.append(score)
        clean.append(("".join(text_clean), scores_clean))

    # descobre o comprimento mais comum entre as predições
    length = Counter(len(t) for t, _ in clean).most_common(1)[0][0]

    # filtra só predições com esse comprimento
    clean = [(t, s) for t, s in clean if len(t) == length]

    final_chars = []
    for pos in range(length):
        # coleta (caractere, confiança) de cada frame nessa posição
        char_scores = []
        for text, scores in clean:
            char = text[pos]
            # scores é um tensor de shape (seq_len,) com a confiança por posição
            score = scores[pos].item() if pos < len(scores) else 0.0
            char_scores.append((char, score))

        # conta votos por caractere
        vote_count = Counter(c for c, _ in char_scores)
        max_votes = max(vote_count.values())
        candidates = [c for c, v in vote_count.items() if v == max_votes]

        if len(candidates) == 1:
            final_chars.append(candidates[0])
        else:
            # desempate: média de confiança entre os empatados
            best_char = max(
                candidates,
                key=lambda c: sum(s for ch, s in char_scores if ch == c) / vote_count[c]
            )
            final_chars.append(best_char)

    return "".join(final_chars)


def cer(gt, pred):
    """CER via distância de Levenshtein normalizada (igual ao experimento retificado vs não retificado)."""
    n, m = len(gt), len(pred)
    dp = list(range(m + 1))
    for i in range(1, n + 1):
        prev = dp[:]
        dp[0] = i
        for j in range(1, m + 1):
            cost = 0 if gt[i-1] == pred[j-1] else 1
            dp[j] = min(dp[j] + 1, dp[j-1] + 1, prev[j-1] + cost)
    return dp[m] / max(n, 1)

print("Funções mvcp() e cer() definidas com sucesso.")

# Fine tuning de parseq tiny com RodoSol original não retificada + LRLPR

Carrega tudo e testa

In [ ]:
# carrega modelos
import torch
import os

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Dispositivo de execução: {device.upper()}")

# caminho para pesos salvos no fine-tuning
parseq_finetuned_path = "/content/drive/MyDrive/<CAMINHO_PESOS>/parseq_tiny_finetuned_rodosol_original_final.pt"

print("Carregando o sistema ParSeq ...")
recognizer_system = torch.hub.load('baudm/parseq', 'parseq_tiny', pretrained=False).to(device)

# carregar pesos fine-tuned dentro do sub-modelo
print(f"Carregando pesos fine-tuned de: {parseq_finetuned_path}")
if os.path.exists(parseq_finetuned_path):
    try:
        # carrega o state_dict (os pesos)
        state_dict = torch.load(parseq_finetuned_path, map_location=device)

        recognizer_system.model.load_state_dict(state_dict)

        print("Pesos fine-tuned aplicados com sucesso em 'recognizer_system.model'.")
    except Exception as e:
        print(f"ERRO ao carregar state_dict: {e}. O modelo continuará com pesos aleatórios.")
else:
    print(f"ERRO: Arquivo de pesos {parseq_finetuned_path} NÃO ENCONTRADO!")
    print("O modelo 'recognizer_system.model' continuará com pesos NÃO TREINADOS.")

# eval = congela comportamento do modelo, deixa estavel
recognizer_system.eval()
print("Reconhecedor (recognizer_system) pronto para inferência.")

In [ ]:
import os
import json
import cv2
from PIL import Image
from torchvision import transforms
from collections import defaultdict
from google.colab.patches import cv2_imshow
from torchvision import transforms as T
import torch

test_base = '/content/base_competicao/LRLPR-26-5opEvJTW/test'
results_by_category = defaultdict(list)
images_shown_count = 0
max_images_to_show = 3
img_size = recognizer_system.hparams.img_size
parseq_transform = transforms.Compose([
    transforms.Resize(img_size, T.InterpolationMode.BICUBIC),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

print("Iniciando reconhecimento...")
for track_name in sorted(os.listdir(test_base)):
    track_path = os.path.join(test_base, track_name)
    if not os.path.isdir(track_path):
        continue

    annotation_path = os.path.join(track_path, 'annotations.json')
    if not os.path.exists(annotation_path):
        print(f"  - Aviso: sem anotação em {track_name}, pulando.")
        continue

    with open(annotation_path, 'r') as f:
        ann = json.load(f)
    gt_plate = ann.get('plate_text', '').upper()
    category = ann.get('plate_layout', 'unknown')
    if not gt_plate:
        continue

    frames = sorted([f for f in os.listdir(track_path)
                     if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
    if not frames:
        continue

    # acumula predições e scores de todos os frames da track
    track_predictions = []  # lista de (texto, tensor_de_scores_por_char)

    for img_name in frames:
        img_path = os.path.join(track_path, img_name)
        image = cv2.imread(img_path)
        if image is None:
            continue

        if images_shown_count < max_images_to_show:
            print(f"  -> Exibindo: {track_name}/{img_name}")
            cv2_imshow(image)
            images_shown_count += 1

        plate_pil = Image.fromarray(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
        plate_tensor = parseq_transform(plate_pil).unsqueeze(0).to(device)

        with torch.no_grad():
            logits = recognizer_system(images=plate_tensor)

        # desempacota texto e scores por caractere
        texts, scores = recognizer_system.tokenizer.decode(logits.cpu())
        track_predictions.append((texts[0], scores[0]))

    # salva todas as predições da track para aplicar MVCP depois
    results_by_category[category].append({
        'ground_truth': gt_plate,
        'track': track_name,
        'predictions': track_predictions  # lista de (texto, scores)
    })

print("\nReconhecimento concluído.")

Análise de resultado (Acurácia MVCP + CER)

In [ ]:
from collections import defaultdict, Counter
import numpy as np

if "historico_modelos" not in globals():
    historico_modelos = {}

def mvcp(predictions):
    """
    predictions: lista de (texto, tensor_de_scores_por_char)
    Retorna a predição final da track via Majority Vote by Character Position.
    """
    if not predictions:
        return ""

    # limpa caracteres não alfanuméricos (ex: '-' em placas BR)
    clean = []
    for text, scores in predictions:
        text_clean = []
        scores_clean = []
        for char, score in zip(text, scores):
            if char.isalnum():
                text_clean.append(char)
                scores_clean.append(score)
        clean.append(("".join(text_clean), scores_clean))

    # descobre o comprimento mais comum entre as predições
    length = Counter(len(t) for t, _ in clean).most_common(1)[0][0]

    # filtra só predições com esse comprimento
    clean = [(t, s) for t, s in clean if len(t) == length]

    final_chars = []
    for pos in range(length):
        # coleta (caractere, confiança) de cada frame nessa posição
        char_scores = []
        for text, scores in clean:
            char = text[pos]
            # scores é um tensor de shape (seq_len,) com a confiança por posição
            score = scores[pos].item() if pos < len(scores) else 0.0
            char_scores.append((char, score))

        # conta votos por caractere
        vote_count = Counter(c for c, _ in char_scores)
        max_votes = max(vote_count.values())
        candidates = [c for c, v in vote_count.items() if v == max_votes]

        if len(candidates) == 1:
            final_chars.append(candidates[0])
        else:
            # desempate: média de confiança entre os empatados
            best_char = max(
                candidates,
                key=lambda c: sum(s for ch, s in char_scores if ch == c) / vote_count[c]
            )
            final_chars.append(best_char)

    return "".join(final_chars)


def cer(gt, pred):
    """CER via distância de Levenshtein normalizada (igual ao experimento retificado vs não retificado)."""
    n, m = len(gt), len(pred)
    dp = list(range(m + 1))
    for i in range(1, n + 1):
        prev = dp[:]
        dp[0] = i
        for j in range(1, m + 1):
            cost = 0 if gt[i-1] == pred[j-1] else 1
            dp[j] = min(dp[j] + 1, dp[j-1] + 1, prev[j-1] + cost)
    return dp[m] / max(n, 1)


# contagem de acertos, totais e CER por categoria
correct_counts = defaultdict(int)
total_counts = defaultdict(int)
cer_by_category = defaultdict(list)
all_cer_vals = []

for category, results in results_by_category.items():
    if not results:
        continue
    for res in results:
        total_counts[category] += 1
        prediction = mvcp(res['predictions'])
        gt = res['ground_truth']

        if prediction == gt:
            correct_counts[category] += 1

        track_cer = cer(gt, prediction)
        cer_by_category[category].append(track_cer)
        all_cer_vals.append(track_cer)

# resultados de Acurácia (Recognition Rate) e CER
print("\n--- Recognition Rate e CER por Track (MVCP) ---")
total_correct_global = sum(correct_counts.values())
total_plates_global = sum(total_counts.values())

cat_metrics = {}
for category in sorted(total_counts.keys()):
    rr = (correct_counts[category] / total_counts[category]) * 100 if total_counts[category] > 0 else 0.0
    mean_cat_cer = np.mean(cer_by_category[category]) * 100 if cer_by_category[category] else 0.0
    cat_metrics[category] = {"acc": rr, "cer": mean_cat_cer}
    print(f"- Categoria '{category}': Acc = {rr:.2f}% ({correct_counts[category]}/{total_counts[category]} tracks corretas) | CER médio = {mean_cat_cer:.2f}%")

global_rr = (total_correct_global / total_plates_global) * 100 if total_plates_global > 0 else 0.0
global_cer = np.mean(all_cer_vals) * 100 if all_cer_vals else 0.0

print("------------------------------------------")
print(f"Recognition Rate GLOBAL: {global_rr:.2f}% ({total_correct_global}/{total_plates_global} tracks corretas)")
print(f"CER médio GLOBAL:        {global_cer:.2f}%")
print(f"CER = 0 (exatas):        {sum(v == 0 for v in all_cer_vals)} tracks")
print(f"CER > 50%:               {sum(v > 0.5 for v in all_cer_vals)} tracks")
print("------------------------------------------")

historico_modelos["Fine tuning de parseq tiny com RodoSol original não retificada + LRLPR"] = {
    "Brazilian_acc": cat_metrics.get("Brazilian", {}).get("acc", 0.0),
    "Brazilian_cer": cat_metrics.get("Brazilian", {}).get("cer", 0.0),
    "Mercosur_acc": cat_metrics.get("Mercosur", {}).get("acc", 0.0),
    "Mercosur_cer": cat_metrics.get("Mercosur", {}).get("cer", 0.0),
    "Global_acc": global_rr,
    "Global_cer": global_cer,
}

# Fine tuning de parseq tiny com RodoSol original degradada sinteticamente + LRLPR

Carrega tudo e testa

In [ ]:
# carrega modelos
import torch
import os

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Dispositivo de execução: {device.upper()}")

# caminho para pesos salvos no fine-tuning
parseq_finetuned_path = "/content/drive/MyDrive/<CAMINHO_PESOS>/parseq_tiny_finetuned_rodosol_degradada_sint_final.pt"

print("Carregando o sistema ParSeq ...")
recognizer_system = torch.hub.load('baudm/parseq', 'parseq_tiny', pretrained=False).to(device)

# carregar pesos fine-tuned dentro do sub-modelo
print(f"Carregando pesos fine-tuned de: {parseq_finetuned_path}")
if os.path.exists(parseq_finetuned_path):
    try:
        # carrega o state_dict (os pesos)
        state_dict = torch.load(parseq_finetuned_path, map_location=device)

        recognizer_system.model.load_state_dict(state_dict)

        print("Pesos fine-tuned aplicados com sucesso em 'recognizer_system.model'.")
    except Exception as e:
        print(f"ERRO ao carregar state_dict: {e}. O modelo continuará com pesos aleatórios.")
else:
    print(f"ERRO: Arquivo de pesos {parseq_finetuned_path} NÃO ENCONTRADO!")
    print("O modelo 'recognizer_system.model' continuará com pesos NÃO TREINADOS.")

# eval = congela comportamento do modelo, deixa estavel
recognizer_system.eval()
print("Reconhecedor (recognizer_system) pronto para inferência.")

In [ ]:
import os
import json
import cv2
from PIL import Image
from torchvision import transforms
from collections import defaultdict
from google.colab.patches import cv2_imshow
from torchvision import transforms as T
import torch

test_base = '/content/base_competicao/LRLPR-26-5opEvJTW/test'
results_by_category = defaultdict(list)
images_shown_count = 0
max_images_to_show = 3
img_size = recognizer_system.hparams.img_size
parseq_transform = transforms.Compose([
    transforms.Resize(img_size, T.InterpolationMode.BICUBIC),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

print("Iniciando reconhecimento...")
for track_name in sorted(os.listdir(test_base)):
    track_path = os.path.join(test_base, track_name)
    if not os.path.isdir(track_path):
        continue

    annotation_path = os.path.join(track_path, 'annotations.json')
    if not os.path.exists(annotation_path):
        print(f"  - Aviso: sem anotação em {track_name}, pulando.")
        continue

    with open(annotation_path, 'r') as f:
        ann = json.load(f)
    gt_plate = ann.get('plate_text', '').upper()
    category = ann.get('plate_layout', 'unknown')
    if not gt_plate:
        continue

    frames = sorted([f for f in os.listdir(track_path)
                     if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
    if not frames:
        continue

    # acumula predições e scores de todos os frames da track
    track_predictions = []  # lista de (texto, tensor_de_scores_por_char)

    for img_name in frames:
        img_path = os.path.join(track_path, img_name)
        image = cv2.imread(img_path)
        if image is None:
            continue

        if images_shown_count < max_images_to_show:
            print(f"  -> Exibindo: {track_name}/{img_name}")
            cv2_imshow(image)
            images_shown_count += 1

        plate_pil = Image.fromarray(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
        plate_tensor = parseq_transform(plate_pil).unsqueeze(0).to(device)

        with torch.no_grad():
            logits = recognizer_system(images=plate_tensor)

        # desempacota texto e scores por caractere
        texts, scores = recognizer_system.tokenizer.decode(logits.cpu())
        track_predictions.append((texts[0], scores[0]))

    # salva todas as predições da track para aplicar MVCP depois
    results_by_category[category].append({
        'ground_truth': gt_plate,
        'track': track_name,
        'predictions': track_predictions  # lista de (texto, scores)
    })

print("\nReconhecimento concluído.")

Análise de resultado (Acurácia MVCP + CER)

In [ ]:
from collections import defaultdict, Counter
import numpy as np

if "historico_modelos" not in globals():
    historico_modelos = {}

def mvcp(predictions):
    """
    predictions: lista de (texto, tensor_de_scores_por_char)
    Retorna a predição final da track via Majority Vote by Character Position.
    """
    if not predictions:
        return ""

    # limpa caracteres não alfanuméricos (ex: '-' em placas BR)
    clean = []
    for text, scores in predictions:
        text_clean = []
        scores_clean = []
        for char, score in zip(text, scores):
            if char.isalnum():
                text_clean.append(char)
                scores_clean.append(score)
        clean.append(("".join(text_clean), scores_clean))

    # descobre o comprimento mais comum entre as predições
    length = Counter(len(t) for t, _ in clean).most_common(1)[0][0]

    # filtra só predições com esse comprimento
    clean = [(t, s) for t, s in clean if len(t) == length]

    final_chars = []
    for pos in range(length):
        # coleta (caractere, confiança) de cada frame nessa posição
        char_scores = []
        for text, scores in clean:
            char = text[pos]
            # scores é um tensor de shape (seq_len,) com a confiança por posição
            score = scores[pos].item() if pos < len(scores) else 0.0
            char_scores.append((char, score))

        # conta votos por caractere
        vote_count = Counter(c for c, _ in char_scores)
        max_votes = max(vote_count.values())
        candidates = [c for c, v in vote_count.items() if v == max_votes]

        if len(candidates) == 1:
            final_chars.append(candidates[0])
        else:
            # desempate: média de confiança entre os empatados
            best_char = max(
                candidates,
                key=lambda c: sum(s for ch, s in char_scores if ch == c) / vote_count[c]
            )
            final_chars.append(best_char)

    return "".join(final_chars)


def cer(gt, pred):
    """CER via distância de Levenshtein normalizada (igual ao experimento retificado vs não retificado)."""
    n, m = len(gt), len(pred)
    dp = list(range(m + 1))
    for i in range(1, n + 1):
        prev = dp[:]
        dp[0] = i
        for j in range(1, m + 1):
            cost = 0 if gt[i-1] == pred[j-1] else 1
            dp[j] = min(dp[j] + 1, dp[j-1] + 1, prev[j-1] + cost)
    return dp[m] / max(n, 1)


# contagem de acertos, totais e CER por categoria
correct_counts = defaultdict(int)
total_counts = defaultdict(int)
cer_by_category = defaultdict(list)
all_cer_vals = []

for category, results in results_by_category.items():
    if not results:
        continue
    for res in results:
        total_counts[category] += 1
        prediction = mvcp(res['predictions'])
        gt = res['ground_truth']

        if prediction == gt:
            correct_counts[category] += 1

        track_cer = cer(gt, prediction)
        cer_by_category[category].append(track_cer)
        all_cer_vals.append(track_cer)

# resultados de Acurácia (Recognition Rate) e CER
print("\n--- Recognition Rate e CER por Track (MVCP) ---")
total_correct_global = sum(correct_counts.values())
total_plates_global = sum(total_counts.values())

cat_metrics = {}
for category in sorted(total_counts.keys()):
    rr = (correct_counts[category] / total_counts[category]) * 100 if total_counts[category] > 0 else 0.0
    mean_cat_cer = np.mean(cer_by_category[category]) * 100 if cer_by_category[category] else 0.0
    cat_metrics[category] = {"acc": rr, "cer": mean_cat_cer}
    print(f"- Categoria '{category}': Acc = {rr:.2f}% ({correct_counts[category]}/{total_counts[category]} tracks corretas) | CER médio = {mean_cat_cer:.2f}%")

global_rr = (total_correct_global / total_plates_global) * 100 if total_plates_global > 0 else 0.0
global_cer = np.mean(all_cer_vals) * 100 if all_cer_vals else 0.0

print("------------------------------------------")
print(f"Recognition Rate GLOBAL: {global_rr:.2f}% ({total_correct_global}/{total_plates_global} tracks corretas)")
print(f"CER médio GLOBAL:        {global_cer:.2f}%")
print(f"CER = 0 (exatas):        {sum(v == 0 for v in all_cer_vals)} tracks")
print(f"CER > 50%:               {sum(v > 0.5 for v in all_cer_vals)} tracks")
print("------------------------------------------")

historico_modelos["Fine tuning de parseq tiny com RodoSol original degradada sinteticamente + LRLPR"] = {
    "Brazilian_acc": cat_metrics.get("Brazilian", {}).get("acc", 0.0),
    "Brazilian_cer": cat_metrics.get("Brazilian", {}).get("cer", 0.0),
    "Mercosur_acc": cat_metrics.get("Mercosur", {}).get("acc", 0.0),
    "Mercosur_cer": cat_metrics.get("Mercosur", {}).get("cer", 0.0),
    "Global_acc": global_rr,
    "Global_cer": global_cer,
}

# Fine tuning de parseq tiny com base de competição pura (baseline)

Carrega tudo e testa

In [ ]:
# carrega modelos
import torch
import os

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Dispositivo de execução: {device.upper()}")

# caminho para pesos salvos no fine-tuning
parseq_finetuned_path = "/content/drive/MyDrive/<CAMINHO_PESOS>/parseq_tiny_finetuned_base_competicao_inalterada.pt"

print("Carregando o sistema ParSeq ...")
recognizer_system = torch.hub.load('baudm/parseq', 'parseq_tiny', pretrained=False).to(device)

# carregar pesos fine-tuned dentro do sub-modelo
print(f"Carregando pesos fine-tuned de: {parseq_finetuned_path}")
if os.path.exists(parseq_finetuned_path):
    try:
        # carrega o state_dict (os pesos)
        state_dict = torch.load(parseq_finetuned_path, map_location=device)

        recognizer_system.model.load_state_dict(state_dict)

        print("Pesos fine-tuned aplicados com sucesso em 'recognizer_system.model'.")
    except Exception as e:
        print(f"ERRO ao carregar state_dict: {e}. O modelo continuará com pesos aleatórios.")
else:
    print(f"ERRO: Arquivo de pesos {parseq_finetuned_path} NÃO ENCONTRADO!")
    print("O modelo 'recognizer_system.model' continuará com pesos NÃO TREINADOS.")

# eval = congela comportamento do modelo, deixa estavel
recognizer_system.eval()
print("Reconhecedor (recognizer_system) pronto para inferência.")

In [ ]:
import os
import json
import cv2
from PIL import Image
from torchvision import transforms
from collections import defaultdict
from google.colab.patches import cv2_imshow
from torchvision import transforms as T
import torch

test_base = '/content/base_competicao/LRLPR-26-5opEvJTW/test'
results_by_category = defaultdict(list)
images_shown_count = 0
max_images_to_show = 3
img_size = recognizer_system.hparams.img_size
parseq_transform = transforms.Compose([
    transforms.Resize(img_size, T.InterpolationMode.BICUBIC),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

print("Iniciando reconhecimento...")
for track_name in sorted(os.listdir(test_base)):
    track_path = os.path.join(test_base, track_name)
    if not os.path.isdir(track_path):
        continue

    annotation_path = os.path.join(track_path, 'annotations.json')
    if not os.path.exists(annotation_path):
        print(f"  - Aviso: sem anotação em {track_name}, pulando.")
        continue

    with open(annotation_path, 'r') as f:
        ann = json.load(f)
    gt_plate = ann.get('plate_text', '').upper()
    category = ann.get('plate_layout', 'unknown')
    if not gt_plate:
        continue

    frames = sorted([f for f in os.listdir(track_path)
                     if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
    if not frames:
        continue

    # acumula predições e scores de todos os frames da track
    track_predictions = []  # lista de (texto, tensor_de_scores_por_char)

    for img_name in frames:
        img_path = os.path.join(track_path, img_name)
        image = cv2.imread(img_path)
        if image is None:
            continue

        if images_shown_count < max_images_to_show:
            print(f"  -> Exibindo: {track_name}/{img_name}")
            cv2_imshow(image)
            images_shown_count += 1

        plate_pil = Image.fromarray(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
        plate_tensor = parseq_transform(plate_pil).unsqueeze(0).to(device)

        with torch.no_grad():
            logits = recognizer_system(images=plate_tensor)

        # desempacota texto e scores por caractere
        texts, scores = recognizer_system.tokenizer.decode(logits.cpu())
        track_predictions.append((texts[0], scores[0]))

    # salva todas as predições da track para aplicar MVCP depois
    results_by_category[category].append({
        'ground_truth': gt_plate,
        'track': track_name,
        'predictions': track_predictions  # lista de (texto, scores)
    })

print("\nReconhecimento concluído.")

Análise de resultado (Acurácia MVCP + CER)

In [ ]:
from collections import defaultdict, Counter
import numpy as np

if "historico_modelos" not in globals():
    historico_modelos = {}

def mvcp(predictions):
    """
    predictions: lista de (texto, tensor_de_scores_por_char)
    Retorna a predição final da track via Majority Vote by Character Position.
    """
    if not predictions:
        return ""

    # limpa caracteres não alfanuméricos (ex: '-' em placas BR)
    clean = []
    for text, scores in predictions:
        text_clean = []
        scores_clean = []
        for char, score in zip(text, scores):
            if char.isalnum():
                text_clean.append(char)
                scores_clean.append(score)
        clean.append(("".join(text_clean), scores_clean))

    # descobre o comprimento mais comum entre as predições
    length = Counter(len(t) for t, _ in clean).most_common(1)[0][0]

    # filtra só predições com esse comprimento
    clean = [(t, s) for t, s in clean if len(t) == length]

    final_chars = []
    for pos in range(length):
        # coleta (caractere, confiança) de cada frame nessa posição
        char_scores = []
        for text, scores in clean:
            char = text[pos]
            # scores é um tensor de shape (seq_len,) com a confiança por posição
            score = scores[pos].item() if pos < len(scores) else 0.0
            char_scores.append((char, score))

        # conta votos por caractere
        vote_count = Counter(c for c, _ in char_scores)
        max_votes = max(vote_count.values())
        candidates = [c for c, v in vote_count.items() if v == max_votes]

        if len(candidates) == 1:
            final_chars.append(candidates[0])
        else:
            # desempate: média de confiança entre os empatados
            best_char = max(
                candidates,
                key=lambda c: sum(s for ch, s in char_scores if ch == c) / vote_count[c]
            )
            final_chars.append(best_char)

    return "".join(final_chars)


def cer(gt, pred):
    """CER via distância de Levenshtein normalizada (igual ao experimento retificado vs não retificado)."""
    n, m = len(gt), len(pred)
    dp = list(range(m + 1))
    for i in range(1, n + 1):
        prev = dp[:]
        dp[0] = i
        for j in range(1, m + 1):
            cost = 0 if gt[i-1] == pred[j-1] else 1
            dp[j] = min(dp[j] + 1, dp[j-1] + 1, prev[j-1] + cost)
    return dp[m] / max(n, 1)


# contagem de acertos, totais e CER por categoria
correct_counts = defaultdict(int)
total_counts = defaultdict(int)
cer_by_category = defaultdict(list)
all_cer_vals = []

for category, results in results_by_category.items():
    if not results:
        continue
    for res in results:
        total_counts[category] += 1
        prediction = mvcp(res['predictions'])
        gt = res['ground_truth']

        if prediction == gt:
            correct_counts[category] += 1

        track_cer = cer(gt, prediction)
        cer_by_category[category].append(track_cer)
        all_cer_vals.append(track_cer)

# resultados de Acurácia (Recognition Rate) e CER
print("\n--- Recognition Rate e CER por Track (MVCP) ---")
total_correct_global = sum(correct_counts.values())
total_plates_global = sum(total_counts.values())

cat_metrics = {}
for category in sorted(total_counts.keys()):
    rr = (correct_counts[category] / total_counts[category]) * 100 if total_counts[category] > 0 else 0.0
    mean_cat_cer = np.mean(cer_by_category[category]) * 100 if cer_by_category[category] else 0.0
    cat_metrics[category] = {"acc": rr, "cer": mean_cat_cer}
    print(f"- Categoria '{category}': Acc = {rr:.2f}% ({correct_counts[category]}/{total_counts[category]} tracks corretas) | CER médio = {mean_cat_cer:.2f}%")

global_rr = (total_correct_global / total_plates_global) * 100 if total_plates_global > 0 else 0.0
global_cer = np.mean(all_cer_vals) * 100 if all_cer_vals else 0.0

print("------------------------------------------")
print(f"Recognition Rate GLOBAL: {global_rr:.2f}% ({total_correct_global}/{total_plates_global} tracks corretas)")
print(f"CER médio GLOBAL:        {global_cer:.2f}%")
print(f"CER = 0 (exatas):        {sum(v == 0 for v in all_cer_vals)} tracks")
print(f"CER > 50%:               {sum(v > 0.5 for v in all_cer_vals)} tracks")
print("------------------------------------------")

historico_modelos["Fine tuning de parseq tiny com base de competição pura (baseline)"] = {
    "Brazilian_acc": cat_metrics.get("Brazilian", {}).get("acc", 0.0),
    "Brazilian_cer": cat_metrics.get("Brazilian", {}).get("cer", 0.0),
    "Mercosur_acc": cat_metrics.get("Mercosur", {}).get("acc", 0.0),
    "Mercosur_cer": cat_metrics.get("Mercosur", {}).get("cer", 0.0),
    "Global_acc": global_rr,
    "Global_cer": global_cer,
}

# Fine tuning de parseq tiny com base de competição mista batch 8 TSIT

Carrega tudo e testa

In [ ]:
# carrega modelos
import torch
import os

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Dispositivo de execução: {device.upper()}")

# caminho para pesos salvos no fine-tuning
parseq_finetuned_path = "/content/drive/MyDrive/<CAMINHO_PESOS>/parseq_tiny_finetuned_base_competicao_mista.pt"

print("Carregando o sistema ParSeq ...")
recognizer_system = torch.hub.load('baudm/parseq', 'parseq_tiny', pretrained=False).to(device)

# carregar pesos fine-tuned dentro do sub-modelo
print(f"Carregando pesos fine-tuned de: {parseq_finetuned_path}")
if os.path.exists(parseq_finetuned_path):
    try:
        # carrega o state_dict (os pesos)
        state_dict = torch.load(parseq_finetuned_path, map_location=device)

        recognizer_system.model.load_state_dict(state_dict)

        print("Pesos fine-tuned aplicados com sucesso em 'recognizer_system.model'.")
    except Exception as e:
        print(f"ERRO ao carregar state_dict: {e}. O modelo continuará com pesos aleatórios.")
else:
    print(f"ERRO: Arquivo de pesos {parseq_finetuned_path} NÃO ENCONTRADO!")
    print("O modelo 'recognizer_system.model' continuará com pesos NÃO TREINADOS.")

# eval = congela comportamento do modelo, deixa estavel
recognizer_system.eval()
print("Reconhecedor (recognizer_system) pronto para inferência.")

In [ ]:
import os
import json
import cv2
from PIL import Image
from torchvision import transforms
from collections import defaultdict
from google.colab.patches import cv2_imshow
from torchvision import transforms as T
import torch

test_base = '/content/base_competicao/LRLPR-26-5opEvJTW/test'
results_by_category = defaultdict(list)
images_shown_count = 0
max_images_to_show = 3
img_size = recognizer_system.hparams.img_size
parseq_transform = transforms.Compose([
    transforms.Resize(img_size, T.InterpolationMode.BICUBIC),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

print("Iniciando reconhecimento...")
for track_name in sorted(os.listdir(test_base)):
    track_path = os.path.join(test_base, track_name)
    if not os.path.isdir(track_path):
        continue

    annotation_path = os.path.join(track_path, 'annotations.json')
    if not os.path.exists(annotation_path):
        print(f"  - Aviso: sem anotação em {track_name}, pulando.")
        continue

    with open(annotation_path, 'r') as f:
        ann = json.load(f)
    gt_plate = ann.get('plate_text', '').upper()
    category = ann.get('plate_layout', 'unknown')
    if not gt_plate:
        continue

    frames = sorted([f for f in os.listdir(track_path)
                     if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
    if not frames:
        continue

    # acumula predições e scores de todos os frames da track
    track_predictions = []  # lista de (texto, tensor_de_scores_por_char)

    for img_name in frames:
        img_path = os.path.join(track_path, img_name)
        image = cv2.imread(img_path)
        if image is None:
            continue

        if images_shown_count < max_images_to_show:
            print(f"  -> Exibindo: {track_name}/{img_name}")
            cv2_imshow(image)
            images_shown_count += 1

        plate_pil = Image.fromarray(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
        plate_tensor = parseq_transform(plate_pil).unsqueeze(0).to(device)

        with torch.no_grad():
            logits = recognizer_system(images=plate_tensor)

        # desempacota texto e scores por caractere
        texts, scores = recognizer_system.tokenizer.decode(logits.cpu())
        track_predictions.append((texts[0], scores[0]))

    # salva todas as predições da track para aplicar MVCP depois
    results_by_category[category].append({
        'ground_truth': gt_plate,
        'track': track_name,
        'predictions': track_predictions  # lista de (texto, scores)
    })

print("\nReconhecimento concluído.")

Análise de resultado (Acurácia MVCP + CER)

In [ ]:
from collections import defaultdict, Counter
import numpy as np

if "historico_modelos" not in globals():
    historico_modelos = {}

def mvcp(predictions):
    """
    predictions: lista de (texto, tensor_de_scores_por_char)
    Retorna a predição final da track via Majority Vote by Character Position.
    """
    if not predictions:
        return ""

    # limpa caracteres não alfanuméricos (ex: '-' em placas BR)
    clean = []
    for text, scores in predictions:
        text_clean = []
        scores_clean = []
        for char, score in zip(text, scores):
            if char.isalnum():
                text_clean.append(char)
                scores_clean.append(score)
        clean.append(("".join(text_clean), scores_clean))

    # descobre o comprimento mais comum entre as predições
    length = Counter(len(t) for t, _ in clean).most_common(1)[0][0]

    # filtra só predições com esse comprimento
    clean = [(t, s) for t, s in clean if len(t) == length]

    final_chars = []
    for pos in range(length):
        # coleta (caractere, confiança) de cada frame nessa posição
        char_scores = []
        for text, scores in clean:
            char = text[pos]
            # scores é um tensor de shape (seq_len,) com a confiança por posição
            score = scores[pos].item() if pos < len(scores) else 0.0
            char_scores.append((char, score))

        # conta votos por caractere
        vote_count = Counter(c for c, _ in char_scores)
        max_votes = max(vote_count.values())
        candidates = [c for c, v in vote_count.items() if v == max_votes]

        if len(candidates) == 1:
            final_chars.append(candidates[0])
        else:
            # desempate: média de confiança entre os empatados
            best_char = max(
                candidates,
                key=lambda c: sum(s for ch, s in char_scores if ch == c) / vote_count[c]
            )
            final_chars.append(best_char)

    return "".join(final_chars)


def cer(gt, pred):
    """CER via distância de Levenshtein normalizada (igual ao experimento retificado vs não retificado)."""
    n, m = len(gt), len(pred)
    dp = list(range(m + 1))
    for i in range(1, n + 1):
        prev = dp[:]
        dp[0] = i
        for j in range(1, m + 1):
            cost = 0 if gt[i-1] == pred[j-1] else 1
            dp[j] = min(dp[j] + 1, dp[j-1] + 1, prev[j-1] + cost)
    return dp[m] / max(n, 1)


# contagem de acertos, totais e CER por categoria
correct_counts = defaultdict(int)
total_counts = defaultdict(int)
cer_by_category = defaultdict(list)
all_cer_vals = []

for category, results in results_by_category.items():
    if not results:
        continue
    for res in results:
        total_counts[category] += 1
        prediction = mvcp(res['predictions'])
        gt = res['ground_truth']

        if prediction == gt:
            correct_counts[category] += 1

        track_cer = cer(gt, prediction)
        cer_by_category[category].append(track_cer)
        all_cer_vals.append(track_cer)

# resultados de Acurácia (Recognition Rate) e CER
print("\n--- Recognition Rate e CER por Track (MVCP) ---")
total_correct_global = sum(correct_counts.values())
total_plates_global = sum(total_counts.values())

cat_metrics = {}
for category in sorted(total_counts.keys()):
    rr = (correct_counts[category] / total_counts[category]) * 100 if total_counts[category] > 0 else 0.0
    mean_cat_cer = np.mean(cer_by_category[category]) * 100 if cer_by_category[category] else 0.0
    cat_metrics[category] = {"acc": rr, "cer": mean_cat_cer}
    print(f"- Categoria '{category}': Acc = {rr:.2f}% ({correct_counts[category]}/{total_counts[category]} tracks corretas) | CER médio = {mean_cat_cer:.2f}%")

global_rr = (total_correct_global / total_plates_global) * 100 if total_plates_global > 0 else 0.0
global_cer = np.mean(all_cer_vals) * 100 if all_cer_vals else 0.0

print("------------------------------------------")
print(f"Recognition Rate GLOBAL: {global_rr:.2f}% ({total_correct_global}/{total_plates_global} tracks corretas)")
print(f"CER médio GLOBAL:        {global_cer:.2f}%")
print(f"CER = 0 (exatas):        {sum(v == 0 for v in all_cer_vals)} tracks")
print(f"CER > 50%:               {sum(v > 0.5 for v in all_cer_vals)} tracks")
print("------------------------------------------")

historico_modelos["Fine tuning de parseq tiny com base de competição mista batch 8 TSIT"] = {
    "Brazilian_acc": cat_metrics.get("Brazilian", {}).get("acc", 0.0),
    "Brazilian_cer": cat_metrics.get("Brazilian", {}).get("cer", 0.0),
    "Mercosur_acc": cat_metrics.get("Mercosur", {}).get("acc", 0.0),
    "Mercosur_cer": cat_metrics.get("Mercosur", {}).get("cer", 0.0),
    "Global_acc": global_rr,
    "Global_cer": global_cer,
}

# Fine tuning de parseq tiny com base de competição mista batch 16 TSIT

Carrega tudo e testa

In [ ]:
# carrega modelos
import torch
import os

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Dispositivo de execução: {device.upper()}")

# caminho para pesos salvos no fine-tuning
parseq_finetuned_path = "/content/drive/MyDrive/<CAMINHO_PESOS>/parseq_tiny_finetuned_base_competicao_mista16.pt"

print("Carregando o sistema ParSeq ...")
recognizer_system = torch.hub.load('baudm/parseq', 'parseq_tiny', pretrained=False).to(device)

# carregar pesos fine-tuned dentro do sub-modelo
print(f"Carregando pesos fine-tuned de: {parseq_finetuned_path}")
if os.path.exists(parseq_finetuned_path):
    try:
        # carrega o state_dict (os pesos)
        state_dict = torch.load(parseq_finetuned_path, map_location=device)

        recognizer_system.model.load_state_dict(state_dict)

        print("Pesos fine-tuned aplicados com sucesso em 'recognizer_system.model'.")
    except Exception as e:
        print(f"ERRO ao carregar state_dict: {e}. O modelo continuará com pesos aleatórios.")
else:
    print(f"ERRO: Arquivo de pesos {parseq_finetuned_path} NÃO ENCONTRADO!")
    print("O modelo 'recognizer_system.model' continuará com pesos NÃO TREINADOS.")

# eval = congela comportamento do modelo, deixa estavel
recognizer_system.eval()
print("Reconhecedor (recognizer_system) pronto para inferência.")

In [ ]:
import os
import json
import cv2
from PIL import Image
from torchvision import transforms
from collections import defaultdict
from google.colab.patches import cv2_imshow
from torchvision import transforms as T
import torch

test_base = '/content/base_competicao/LRLPR-26-5opEvJTW/test'
results_by_category = defaultdict(list)
images_shown_count = 0
max_images_to_show = 3
img_size = recognizer_system.hparams.img_size
parseq_transform = transforms.Compose([
    transforms.Resize(img_size, T.InterpolationMode.BICUBIC),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

print("Iniciando reconhecimento...")
for track_name in sorted(os.listdir(test_base)):
    track_path = os.path.join(test_base, track_name)
    if not os.path.isdir(track_path):
        continue

    annotation_path = os.path.join(track_path, 'annotations.json')
    if not os.path.exists(annotation_path):
        print(f"  - Aviso: sem anotação em {track_name}, pulando.")
        continue

    with open(annotation_path, 'r') as f:
        ann = json.load(f)
    gt_plate = ann.get('plate_text', '').upper()
    category = ann.get('plate_layout', 'unknown')
    if not gt_plate:
        continue

    frames = sorted([f for f in os.listdir(track_path)
                     if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
    if not frames:
        continue

    # acumula predições e scores de todos os frames da track
    track_predictions = []  # lista de (texto, tensor_de_scores_por_char)

    for img_name in frames:
        img_path = os.path.join(track_path, img_name)
        image = cv2.imread(img_path)
        if image is None:
            continue

        if images_shown_count < max_images_to_show:
            print(f"  -> Exibindo: {track_name}/{img_name}")
            cv2_imshow(image)
            images_shown_count += 1

        plate_pil = Image.fromarray(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
        plate_tensor = parseq_transform(plate_pil).unsqueeze(0).to(device)

        with torch.no_grad():
            logits = recognizer_system(images=plate_tensor)

        # desempacota texto e scores por caractere
        texts, scores = recognizer_system.tokenizer.decode(logits.cpu())
        track_predictions.append((texts[0], scores[0]))

    # salva todas as predições da track para aplicar MVCP depois
    results_by_category[category].append({
        'ground_truth': gt_plate,
        'track': track_name,
        'predictions': track_predictions  # lista de (texto, scores)
    })

print("\nReconhecimento concluído.")

Análise de resultado (Acurácia MVCP + CER)

In [ ]:
from collections import defaultdict, Counter
import numpy as np

if "historico_modelos" not in globals():
    historico_modelos = {}

def mvcp(predictions):
    """
    predictions: lista de (texto, tensor_de_scores_por_char)
    Retorna a predição final da track via Majority Vote by Character Position.
    """
    if not predictions:
        return ""

    # limpa caracteres não alfanuméricos (ex: '-' em placas BR)
    clean = []
    for text, scores in predictions:
        text_clean = []
        scores_clean = []
        for char, score in zip(text, scores):
            if char.isalnum():
                text_clean.append(char)
                scores_clean.append(score)
        clean.append(("".join(text_clean), scores_clean))

    # descobre o comprimento mais comum entre as predições
    length = Counter(len(t) for t, _ in clean).most_common(1)[0][0]

    # filtra só predições com esse comprimento
    clean = [(t, s) for t, s in clean if len(t) == length]

    final_chars = []
    for pos in range(length):
        # coleta (caractere, confiança) de cada frame nessa posição
        char_scores = []
        for text, scores in clean:
            char = text[pos]
            # scores é um tensor de shape (seq_len,) com a confiança por posição
            score = scores[pos].item() if pos < len(scores) else 0.0
            char_scores.append((char, score))

        # conta votos por caractere
        vote_count = Counter(c for c, _ in char_scores)
        max_votes = max(vote_count.values())
        candidates = [c for c, v in vote_count.items() if v == max_votes]

        if len(candidates) == 1:
            final_chars.append(candidates[0])
        else:
            # desempate: média de confiança entre os empatados
            best_char = max(
                candidates,
                key=lambda c: sum(s for ch, s in char_scores if ch == c) / vote_count[c]
            )
            final_chars.append(best_char)

    return "".join(final_chars)


def cer(gt, pred):
    """CER via distância de Levenshtein normalizada (igual ao experimento retificado vs não retificado)."""
    n, m = len(gt), len(pred)
    dp = list(range(m + 1))
    for i in range(1, n + 1):
        prev = dp[:]
        dp[0] = i
        for j in range(1, m + 1):
            cost = 0 if gt[i-1] == pred[j-1] else 1
            dp[j] = min(dp[j] + 1, dp[j-1] + 1, prev[j-1] + cost)
    return dp[m] / max(n, 1)


# contagem de acertos, totais e CER por categoria
correct_counts = defaultdict(int)
total_counts = defaultdict(int)
cer_by_category = defaultdict(list)
all_cer_vals = []

for category, results in results_by_category.items():
    if not results:
        continue
    for res in results:
        total_counts[category] += 1
        prediction = mvcp(res['predictions'])
        gt = res['ground_truth']

        if prediction == gt:
            correct_counts[category] += 1

        track_cer = cer(gt, prediction)
        cer_by_category[category].append(track_cer)
        all_cer_vals.append(track_cer)

# resultados de Acurácia (Recognition Rate) e CER
print("\n--- Recognition Rate e CER por Track (MVCP) ---")
total_correct_global = sum(correct_counts.values())
total_plates_global = sum(total_counts.values())

cat_metrics = {}
for category in sorted(total_counts.keys()):
    rr = (correct_counts[category] / total_counts[category]) * 100 if total_counts[category] > 0 else 0.0
    mean_cat_cer = np.mean(cer_by_category[category]) * 100 if cer_by_category[category] else 0.0
    cat_metrics[category] = {"acc": rr, "cer": mean_cat_cer}
    print(f"- Categoria '{category}': Acc = {rr:.2f}% ({correct_counts[category]}/{total_counts[category]} tracks corretas) | CER médio = {mean_cat_cer:.2f}%")

global_rr = (total_correct_global / total_plates_global) * 100 if total_plates_global > 0 else 0.0
global_cer = np.mean(all_cer_vals) * 100 if all_cer_vals else 0.0

print("------------------------------------------")
print(f"Recognition Rate GLOBAL: {global_rr:.2f}% ({total_correct_global}/{total_plates_global} tracks corretas)")
print(f"CER médio GLOBAL:        {global_cer:.2f}%")
print(f"CER = 0 (exatas):        {sum(v == 0 for v in all_cer_vals)} tracks")
print(f"CER > 50%:               {sum(v > 0.5 for v in all_cer_vals)} tracks")
print("------------------------------------------")

historico_modelos["Fine tuning de parseq tiny com base de competição mista batch 16 TSIT"] = {
    "Brazilian_acc": cat_metrics.get("Brazilian", {}).get("acc", 0.0),
    "Brazilian_cer": cat_metrics.get("Brazilian", {}).get("cer", 0.0),
    "Mercosur_acc": cat_metrics.get("Mercosur", {}).get("acc", 0.0),
    "Mercosur_cer": cat_metrics.get("Mercosur", {}).get("cer", 0.0),
    "Global_acc": global_rr,
    "Global_cer": global_cer,
}

# Fine tuning de parseq tiny com base de competição mista batch 16 DMT

Carrega tudo e testa

In [ ]:
# carrega modelos
import torch
import os

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Dispositivo de execução: {device.upper()}")

# caminho para pesos salvos no fine-tuning
parseq_finetuned_path = "/content/drive/MyDrive/<CAMINHO_PESOS>/parseq_tiny_finetuned_base_competicao_mista16_dmt.pt"

print("Carregando o sistema ParSeq ...")
recognizer_system = torch.hub.load('baudm/parseq', 'parseq_tiny', pretrained=False).to(device)

# carregar pesos fine-tuned dentro do sub-modelo
print(f"Carregando pesos fine-tuned de: {parseq_finetuned_path}")
if os.path.exists(parseq_finetuned_path):
    try:
        # carrega o state_dict (os pesos)
        state_dict = torch.load(parseq_finetuned_path, map_location=device)

        recognizer_system.model.load_state_dict(state_dict)

        print("Pesos fine-tuned aplicados com sucesso em 'recognizer_system.model'.")
    except Exception as e:
        print(f"ERRO ao carregar state_dict: {e}. O modelo continuará com pesos aleatórios.")
else:
    print(f"ERRO: Arquivo de pesos {parseq_finetuned_path} NÃO ENCONTRADO!")
    print("O modelo 'recognizer_system.model' continuará com pesos NÃO TREINADOS.")

# eval = congela comportamento do modelo, deixa estavel
recognizer_system.eval()
print("Reconhecedor (recognizer_system) pronto para inferência.")

In [ ]:
import os
import json
import cv2
from PIL import Image
from torchvision import transforms
from collections import defaultdict
from google.colab.patches import cv2_imshow
from torchvision import transforms as T
import torch

test_base = '/content/base_competicao/LRLPR-26-5opEvJTW/test'
results_by_category = defaultdict(list)
images_shown_count = 0
max_images_to_show = 3
img_size = recognizer_system.hparams.img_size
parseq_transform = transforms.Compose([
    transforms.Resize(img_size, T.InterpolationMode.BICUBIC),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

print("Iniciando reconhecimento...")
for track_name in sorted(os.listdir(test_base)):
    track_path = os.path.join(test_base, track_name)
    if not os.path.isdir(track_path):
        continue

    annotation_path = os.path.join(track_path, 'annotations.json')
    if not os.path.exists(annotation_path):
        print(f"  - Aviso: sem anotação em {track_name}, pulando.")
        continue

    with open(annotation_path, 'r') as f:
        ann = json.load(f)
    gt_plate = ann.get('plate_text', '').upper()
    category = ann.get('plate_layout', 'unknown')
    if not gt_plate:
        continue

    frames = sorted([f for f in os.listdir(track_path)
                     if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
    if not frames:
        continue

    # acumula predições e scores de todos os frames da track
    track_predictions = []  # lista de (texto, tensor_de_scores_por_char)

    for img_name in frames:
        img_path = os.path.join(track_path, img_name)
        image = cv2.imread(img_path)
        if image is None:
            continue

        if images_shown_count < max_images_to_show:
            print(f"  -> Exibindo: {track_name}/{img_name}")
            cv2_imshow(image)
            images_shown_count += 1

        plate_pil = Image.fromarray(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
        plate_tensor = parseq_transform(plate_pil).unsqueeze(0).to(device)

        with torch.no_grad():
            logits = recognizer_system(images=plate_tensor)

        # desempacota texto e scores por caractere
        texts, scores = recognizer_system.tokenizer.decode(logits.cpu())
        track_predictions.append((texts[0], scores[0]))

    # salva todas as predições da track para aplicar MVCP depois
    results_by_category[category].append({
        'ground_truth': gt_plate,
        'track': track_name,
        'predictions': track_predictions  # lista de (texto, scores)
    })

print("\nReconhecimento concluído.")

Análise de resultado (Acurácia MVCP + CER)

In [ ]:
from collections import defaultdict, Counter
import numpy as np

if "historico_modelos" not in globals():
    historico_modelos = {}

def mvcp(predictions):
    """
    predictions: lista de (texto, tensor_de_scores_por_char)
    Retorna a predição final da track via Majority Vote by Character Position.
    """
    if not predictions:
        return ""

    # limpa caracteres não alfanuméricos (ex: '-' em placas BR)
    clean = []
    for text, scores in predictions:
        text_clean = []
        scores_clean = []
        for char, score in zip(text, scores):
            if char.isalnum():
                text_clean.append(char)
                scores_clean.append(score)
        clean.append(("".join(text_clean), scores_clean))

    # descobre o comprimento mais comum entre as predições
    length = Counter(len(t) for t, _ in clean).most_common(1)[0][0]

    # filtra só predições com esse comprimento
    clean = [(t, s) for t, s in clean if len(t) == length]

    final_chars = []
    for pos in range(length):
        # coleta (caractere, confiança) de cada frame nessa posição
        char_scores = []
        for text, scores in clean:
            char = text[pos]
            # scores é um tensor de shape (seq_len,) com a confiança por posição
            score = scores[pos].item() if pos < len(scores) else 0.0
            char_scores.append((char, score))

        # conta votos por caractere
        vote_count = Counter(c for c, _ in char_scores)
        max_votes = max(vote_count.values())
        candidates = [c for c, v in vote_count.items() if v == max_votes]

        if len(candidates) == 1:
            final_chars.append(candidates[0])
        else:
            # desempate: média de confiança entre os empatados
            best_char = max(
                candidates,
                key=lambda c: sum(s for ch, s in char_scores if ch == c) / vote_count[c]
            )
            final_chars.append(best_char)

    return "".join(final_chars)


def cer(gt, pred):
    """CER via distância de Levenshtein normalizada (igual ao experimento retificado vs não retificado)."""
    n, m = len(gt), len(pred)
    dp = list(range(m + 1))
    for i in range(1, n + 1):
        prev = dp[:]
        dp[0] = i
        for j in range(1, m + 1):
            cost = 0 if gt[i-1] == pred[j-1] else 1
            dp[j] = min(dp[j] + 1, dp[j-1] + 1, prev[j-1] + cost)
    return dp[m] / max(n, 1)


# contagem de acertos, totais e CER por categoria
correct_counts = defaultdict(int)
total_counts = defaultdict(int)
cer_by_category = defaultdict(list)
all_cer_vals = []

for category, results in results_by_category.items():
    if not results:
        continue
    for res in results:
        total_counts[category] += 1
        prediction = mvcp(res['predictions'])
        gt = res['ground_truth']

        if prediction == gt:
            correct_counts[category] += 1

        track_cer = cer(gt, prediction)
        cer_by_category[category].append(track_cer)
        all_cer_vals.append(track_cer)

# resultados de Acurácia (Recognition Rate) e CER
print("\n--- Recognition Rate e CER por Track (MVCP) ---")
total_correct_global = sum(correct_counts.values())
total_plates_global = sum(total_counts.values())

cat_metrics = {}
for category in sorted(total_counts.keys()):
    rr = (correct_counts[category] / total_counts[category]) * 100 if total_counts[category] > 0 else 0.0
    mean_cat_cer = np.mean(cer_by_category[category]) * 100 if cer_by_category[category] else 0.0
    cat_metrics[category] = {"acc": rr, "cer": mean_cat_cer}
    print(f"- Categoria '{category}': Acc = {rr:.2f}% ({correct_counts[category]}/{total_counts[category]} tracks corretas) | CER médio = {mean_cat_cer:.2f}%")

global_rr = (total_correct_global / total_plates_global) * 100 if total_plates_global > 0 else 0.0
global_cer = np.mean(all_cer_vals) * 100 if all_cer_vals else 0.0

print("------------------------------------------")
print(f"Recognition Rate GLOBAL: {global_rr:.2f}% ({total_correct_global}/{total_plates_global} tracks corretas)")
print(f"CER médio GLOBAL:        {global_cer:.2f}%")
print(f"CER = 0 (exatas):        {sum(v == 0 for v in all_cer_vals)} tracks")
print(f"CER > 50%:               {sum(v > 0.5 for v in all_cer_vals)} tracks")
print("------------------------------------------")

historico_modelos["Fine tuning de parseq tiny com base de competição mista batch 16 DMT"] = {
    "Brazilian_acc": cat_metrics.get("Brazilian", {}).get("acc", 0.0),
    "Brazilian_cer": cat_metrics.get("Brazilian", {}).get("cer", 0.0),
    "Mercosur_acc": cat_metrics.get("Mercosur", {}).get("acc", 0.0),
    "Mercosur_cer": cat_metrics.get("Mercosur", {}).get("cer", 0.0),
    "Global_acc": global_rr,
    "Global_cer": global_cer,
}

# Fine tuning de parseq tiny com base de competição mista batch 8 DMT

Carrega tudo e testa

In [ ]:
# carrega modelos
import torch
import os

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Dispositivo de execução: {device.upper()}")

# caminho para pesos salvos no fine-tuning
parseq_finetuned_path = "/content/drive/MyDrive/<CAMINHO_PESOS>/parseq_tiny_finetuned_base_competicao_mista8_dmt.pt"

print("Carregando o sistema ParSeq ...")
recognizer_system = torch.hub.load('baudm/parseq', 'parseq_tiny', pretrained=False).to(device)

# carregar pesos fine-tuned dentro do sub-modelo
print(f"Carregando pesos fine-tuned de: {parseq_finetuned_path}")
if os.path.exists(parseq_finetuned_path):
    try:
        # carrega o state_dict (os pesos)
        state_dict = torch.load(parseq_finetuned_path, map_location=device)

        recognizer_system.model.load_state_dict(state_dict)

        print("Pesos fine-tuned aplicados com sucesso em 'recognizer_system.model'.")
    except Exception as e:
        print(f"ERRO ao carregar state_dict: {e}. O modelo continuará com pesos aleatórios.")
else:
    print(f"ERRO: Arquivo de pesos {parseq_finetuned_path} NÃO ENCONTRADO!")
    print("O modelo 'recognizer_system.model' continuará com pesos NÃO TREINADOS.")

# eval = congela comportamento do modelo, deixa estavel
recognizer_system.eval()
print("Reconhecedor (recognizer_system) pronto para inferência.")

In [ ]:
import os
import json
import cv2
from PIL import Image
from torchvision import transforms
from collections import defaultdict
from google.colab.patches import cv2_imshow
from torchvision import transforms as T
import torch

test_base = '/content/base_competicao/LRLPR-26-5opEvJTW/test'
results_by_category = defaultdict(list)
images_shown_count = 0
max_images_to_show = 3
img_size = recognizer_system.hparams.img_size
parseq_transform = transforms.Compose([
    transforms.Resize(img_size, T.InterpolationMode.BICUBIC),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

print("Iniciando reconhecimento...")
for track_name in sorted(os.listdir(test_base)):
    track_path = os.path.join(test_base, track_name)
    if not os.path.isdir(track_path):
        continue

    annotation_path = os.path.join(track_path, 'annotations.json')
    if not os.path.exists(annotation_path):
        print(f"  - Aviso: sem anotação em {track_name}, pulando.")
        continue

    with open(annotation_path, 'r') as f:
        ann = json.load(f)
    gt_plate = ann.get('plate_text', '').upper()
    category = ann.get('plate_layout', 'unknown')
    if not gt_plate:
        continue

    frames = sorted([f for f in os.listdir(track_path)
                     if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
    if not frames:
        continue

    # acumula predições e scores de todos os frames da track
    track_predictions = []  # lista de (texto, tensor_de_scores_por_char)

    for img_name in frames:
        img_path = os.path.join(track_path, img_name)
        image = cv2.imread(img_path)
        if image is None:
            continue

        if images_shown_count < max_images_to_show:
            print(f"  -> Exibindo: {track_name}/{img_name}")
            cv2_imshow(image)
            images_shown_count += 1

        plate_pil = Image.fromarray(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
        plate_tensor = parseq_transform(plate_pil).unsqueeze(0).to(device)

        with torch.no_grad():
            logits = recognizer_system(images=plate_tensor)

        # desempacota texto e scores por caractere
        texts, scores = recognizer_system.tokenizer.decode(logits.cpu())
        track_predictions.append((texts[0], scores[0]))

    # salva todas as predições da track para aplicar MVCP depois
    results_by_category[category].append({
        'ground_truth': gt_plate,
        'track': track_name,
        'predictions': track_predictions  # lista de (texto, scores)
    })

print("\nReconhecimento concluído.")

Análise de resultado (Acurácia MVCP + CER)

In [ ]:
from collections import defaultdict, Counter
import numpy as np

if "historico_modelos" not in globals():
    historico_modelos = {}

def mvcp(predictions):
    """
    predictions: lista de (texto, tensor_de_scores_por_char)
    Retorna a predição final da track via Majority Vote by Character Position.
    """
    if not predictions:
        return ""

    # limpa caracteres não alfanuméricos (ex: '-' em placas BR)
    clean = []
    for text, scores in predictions:
        text_clean = []
        scores_clean = []
        for char, score in zip(text, scores):
            if char.isalnum():
                text_clean.append(char)
                scores_clean.append(score)
        clean.append(("".join(text_clean), scores_clean))

    # descobre o comprimento mais comum entre as predições
    length = Counter(len(t) for t, _ in clean).most_common(1)[0][0]

    # filtra só predições com esse comprimento
    clean = [(t, s) for t, s in clean if len(t) == length]

    final_chars = []
    for pos in range(length):
        # coleta (caractere, confiança) de cada frame nessa posição
        char_scores = []
        for text, scores in clean:
            char = text[pos]
            # scores é um tensor de shape (seq_len,) com a confiança por posição
            score = scores[pos].item() if pos < len(scores) else 0.0
            char_scores.append((char, score))

        # conta votos por caractere
        vote_count = Counter(c for c, _ in char_scores)
        max_votes = max(vote_count.values())
        candidates = [c for c, v in vote_count.items() if v == max_votes]

        if len(candidates) == 1:
            final_chars.append(candidates[0])
        else:
            # desempate: média de confiança entre os empatados
            best_char = max(
                candidates,
                key=lambda c: sum(s for ch, s in char_scores if ch == c) / vote_count[c]
            )
            final_chars.append(best_char)

    return "".join(final_chars)


def cer(gt, pred):
    """CER via distância de Levenshtein normalizada (igual ao experimento retificado vs não retificado)."""
    n, m = len(gt), len(pred)
    dp = list(range(m + 1))
    for i in range(1, n + 1):
        prev = dp[:]
        dp[0] = i
        for j in range(1, m + 1):
            cost = 0 if gt[i-1] == pred[j-1] else 1
            dp[j] = min(dp[j] + 1, dp[j-1] + 1, prev[j-1] + cost)
    return dp[m] / max(n, 1)


# contagem de acertos, totais e CER por categoria
correct_counts = defaultdict(int)
total_counts = defaultdict(int)
cer_by_category = defaultdict(list)
all_cer_vals = []

for category, results in results_by_category.items():
    if not results:
        continue
    for res in results:
        total_counts[category] += 1
        prediction = mvcp(res['predictions'])
        gt = res['ground_truth']

        if prediction == gt:
            correct_counts[category] += 1

        track_cer = cer(gt, prediction)
        cer_by_category[category].append(track_cer)
        all_cer_vals.append(track_cer)

# resultados de Acurácia (Recognition Rate) e CER
print("\n--- Recognition Rate e CER por Track (MVCP) ---")
total_correct_global = sum(correct_counts.values())
total_plates_global = sum(total_counts.values())

cat_metrics = {}
for category in sorted(total_counts.keys()):
    rr = (correct_counts[category] / total_counts[category]) * 100 if total_counts[category] > 0 else 0.0
    mean_cat_cer = np.mean(cer_by_category[category]) * 100 if cer_by_category[category] else 0.0
    cat_metrics[category] = {"acc": rr, "cer": mean_cat_cer}
    print(f"- Categoria '{category}': Acc = {rr:.2f}% ({correct_counts[category]}/{total_counts[category]} tracks corretas) | CER médio = {mean_cat_cer:.2f}%")

global_rr = (total_correct_global / total_plates_global) * 100 if total_plates_global > 0 else 0.0
global_cer = np.mean(all_cer_vals) * 100 if all_cer_vals else 0.0

print("------------------------------------------")
print(f"Recognition Rate GLOBAL: {global_rr:.2f}% ({total_correct_global}/{total_plates_global} tracks corretas)")
print(f"CER médio GLOBAL:        {global_cer:.2f}%")
print(f"CER = 0 (exatas):        {sum(v == 0 for v in all_cer_vals)} tracks")
print(f"CER > 50%:               {sum(v > 0.5 for v in all_cer_vals)} tracks")
print("------------------------------------------")

historico_modelos["Fine tuning de parseq tiny com base de competição mista batch 8 DMT"] = {
    "Brazilian_acc": cat_metrics.get("Brazilian", {}).get("acc", 0.0),
    "Brazilian_cer": cat_metrics.get("Brazilian", {}).get("cer", 0.0),
    "Mercosur_acc": cat_metrics.get("Mercosur", {}).get("acc", 0.0),
    "Mercosur_cer": cat_metrics.get("Mercosur", {}).get("cer", 0.0),
    "Global_acc": global_rr,
    "Global_cer": global_cer,
}

# Fine tuning de parseq tiny com base de competição mista cycle gan turbo 25000 passos retificados

Carrega tudo e testa

In [ ]:
# carrega modelos
import torch
import os

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Dispositivo de execução: {device.upper()}")

# caminho para pesos salvos no fine-tuning
parseq_finetuned_path = "/content/drive/MyDrive/<CAMINHO_PESOS>/parseq_tiny_finetuned_base_competicao_mista_cyle_rodosol_retificado_final.pt"

print("Carregando o sistema ParSeq ...")
recognizer_system = torch.hub.load('baudm/parseq', 'parseq_tiny', pretrained=False).to(device)

# carregar pesos fine-tuned dentro do sub-modelo
print(f"Carregando pesos fine-tuned de: {parseq_finetuned_path}")
if os.path.exists(parseq_finetuned_path):
    try:
        # carrega o state_dict (os pesos)
        state_dict = torch.load(parseq_finetuned_path, map_location=device)

        recognizer_system.model.load_state_dict(state_dict)

        print("Pesos fine-tuned aplicados com sucesso em 'recognizer_system.model'.")
    except Exception as e:
        print(f"ERRO ao carregar state_dict: {e}. O modelo continuará com pesos aleatórios.")
else:
    print(f"ERRO: Arquivo de pesos {parseq_finetuned_path} NÃO ENCONTRADO!")
    print("O modelo 'recognizer_system.model' continuará com pesos NÃO TREINADOS.")

# eval = congela comportamento do modelo, deixa estavel
recognizer_system.eval()
print("Reconhecedor (recognizer_system) pronto para inferência.")

In [ ]:
import os
import json
import cv2
from PIL import Image
from torchvision import transforms
from collections import defaultdict
from google.colab.patches import cv2_imshow
from torchvision import transforms as T
import torch

test_base = '/content/base_competicao/LRLPR-26-5opEvJTW/test'
results_by_category = defaultdict(list)
images_shown_count = 0
max_images_to_show = 3
img_size = recognizer_system.hparams.img_size
parseq_transform = transforms.Compose([
    transforms.Resize(img_size, T.InterpolationMode.BICUBIC),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

print("Iniciando reconhecimento...")
for track_name in sorted(os.listdir(test_base)):
    track_path = os.path.join(test_base, track_name)
    if not os.path.isdir(track_path):
        continue

    annotation_path = os.path.join(track_path, 'annotations.json')
    if not os.path.exists(annotation_path):
        print(f"  - Aviso: sem anotação em {track_name}, pulando.")
        continue

    with open(annotation_path, 'r') as f:
        ann = json.load(f)
    gt_plate = ann.get('plate_text', '').upper()
    category = ann.get('plate_layout', 'unknown')
    if not gt_plate:
        continue

    frames = sorted([f for f in os.listdir(track_path)
                     if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
    if not frames:
        continue

    # acumula predições e scores de todos os frames da track
    track_predictions = []  # lista de (texto, tensor_de_scores_por_char)

    for img_name in frames:
        img_path = os.path.join(track_path, img_name)
        image = cv2.imread(img_path)
        if image is None:
            continue

        if images_shown_count < max_images_to_show:
            print(f"  -> Exibindo: {track_name}/{img_name}")
            cv2_imshow(image)
            images_shown_count += 1

        plate_pil = Image.fromarray(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
        plate_tensor = parseq_transform(plate_pil).unsqueeze(0).to(device)

        with torch.no_grad():
            logits = recognizer_system(images=plate_tensor)

        # desempacota texto e scores por caractere
        texts, scores = recognizer_system.tokenizer.decode(logits.cpu())
        track_predictions.append((texts[0], scores[0]))

    # salva todas as predições da track para aplicar MVCP depois
    results_by_category[category].append({
        'ground_truth': gt_plate,
        'track': track_name,
        'predictions': track_predictions  # lista de (texto, scores)
    })

print("\nReconhecimento concluído.")

Análise de resultado (Acurácia MVCP + CER)

In [ ]:
from collections import defaultdict, Counter
import numpy as np

if "historico_modelos" not in globals():
    historico_modelos = {}

def mvcp(predictions):
    """
    predictions: lista de (texto, tensor_de_scores_por_char)
    Retorna a predição final da track via Majority Vote by Character Position.
    """
    if not predictions:
        return ""

    # limpa caracteres não alfanuméricos (ex: '-' em placas BR)
    clean = []
    for text, scores in predictions:
        text_clean = []
        scores_clean = []
        for char, score in zip(text, scores):
            if char.isalnum():
                text_clean.append(char)
                scores_clean.append(score)
        clean.append(("".join(text_clean), scores_clean))

    # descobre o comprimento mais comum entre as predições
    length = Counter(len(t) for t, _ in clean).most_common(1)[0][0]

    # filtra só predições com esse comprimento
    clean = [(t, s) for t, s in clean if len(t) == length]

    final_chars = []
    for pos in range(length):
        # coleta (caractere, confiança) de cada frame nessa posição
        char_scores = []
        for text, scores in clean:
            char = text[pos]
            # scores é um tensor de shape (seq_len,) com a confiança por posição
            score = scores[pos].item() if pos < len(scores) else 0.0
            char_scores.append((char, score))

        # conta votos por caractere
        vote_count = Counter(c for c, _ in char_scores)
        max_votes = max(vote_count.values())
        candidates = [c for c, v in vote_count.items() if v == max_votes]

        if len(candidates) == 1:
            final_chars.append(candidates[0])
        else:
            # desempate: média de confiança entre os empatados
            best_char = max(
                candidates,
                key=lambda c: sum(s for ch, s in char_scores if ch == c) / vote_count[c]
            )
            final_chars.append(best_char)

    return "".join(final_chars)


def cer(gt, pred):
    """CER via distância de Levenshtein normalizada (igual ao experimento retificado vs não retificado)."""
    n, m = len(gt), len(pred)
    dp = list(range(m + 1))
    for i in range(1, n + 1):
        prev = dp[:]
        dp[0] = i
        for j in range(1, m + 1):
            cost = 0 if gt[i-1] == pred[j-1] else 1
            dp[j] = min(dp[j] + 1, dp[j-1] + 1, prev[j-1] + cost)
    return dp[m] / max(n, 1)


# contagem de acertos, totais e CER por categoria
correct_counts = defaultdict(int)
total_counts = defaultdict(int)
cer_by_category = defaultdict(list)
all_cer_vals = []

for category, results in results_by_category.items():
    if not results:
        continue
    for res in results:
        total_counts[category] += 1
        prediction = mvcp(res['predictions'])
        gt = res['ground_truth']

        if prediction == gt:
            correct_counts[category] += 1

        track_cer = cer(gt, prediction)
        cer_by_category[category].append(track_cer)
        all_cer_vals.append(track_cer)

# resultados de Acurácia (Recognition Rate) e CER
print("\n--- Recognition Rate e CER por Track (MVCP) ---")
total_correct_global = sum(correct_counts.values())
total_plates_global = sum(total_counts.values())

cat_metrics = {}
for category in sorted(total_counts.keys()):
    rr = (correct_counts[category] / total_counts[category]) * 100 if total_counts[category] > 0 else 0.0
    mean_cat_cer = np.mean(cer_by_category[category]) * 100 if cer_by_category[category] else 0.0
    cat_metrics[category] = {"acc": rr, "cer": mean_cat_cer}
    print(f"- Categoria '{category}': Acc = {rr:.2f}% ({correct_counts[category]}/{total_counts[category]} tracks corretas) | CER médio = {mean_cat_cer:.2f}%")

global_rr = (total_correct_global / total_plates_global) * 100 if total_plates_global > 0 else 0.0
global_cer = np.mean(all_cer_vals) * 100 if all_cer_vals else 0.0

print("------------------------------------------")
print(f"Recognition Rate GLOBAL: {global_rr:.2f}% ({total_correct_global}/{total_plates_global} tracks corretas)")
print(f"CER médio GLOBAL:        {global_cer:.2f}%")
print(f"CER = 0 (exatas):        {sum(v == 0 for v in all_cer_vals)} tracks")
print(f"CER > 50%:               {sum(v > 0.5 for v in all_cer_vals)} tracks")
print("------------------------------------------")

historico_modelos["Fine tuning de parseq tiny com base de competição mista cycle gan turbo 25000 passos retificados"] = {
    "Brazilian_acc": cat_metrics.get("Brazilian", {}).get("acc", 0.0),
    "Brazilian_cer": cat_metrics.get("Brazilian", {}).get("cer", 0.0),
    "Mercosur_acc": cat_metrics.get("Mercosur", {}).get("acc", 0.0),
    "Mercosur_cer": cat_metrics.get("Mercosur", {}).get("cer", 0.0),
    "Global_acc": global_rr,
    "Global_cer": global_cer,
}

# Fine tuning de parseq tiny com base de competição mista cycle gan turbo 25000 passos NAO retificados

Carrega tudo e testa

In [ ]:
# carrega modelos
import torch
import os

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Dispositivo de execução: {device.upper()}")

# caminho para pesos salvos no fine-tuning
parseq_finetuned_path = "/content/drive/MyDrive/<CAMINHO_PESOS>/parseq_tiny_finetuned_base_competicao_mista_cyle_rodosol_n_retificado_final.pt"

print("Carregando o sistema ParSeq ...")
recognizer_system = torch.hub.load('baudm/parseq', 'parseq_tiny', pretrained=False).to(device)

# carregar pesos fine-tuned dentro do sub-modelo
print(f"Carregando pesos fine-tuned de: {parseq_finetuned_path}")
if os.path.exists(parseq_finetuned_path):
    try:
        # carrega o state_dict (os pesos)
        state_dict = torch.load(parseq_finetuned_path, map_location=device)

        recognizer_system.model.load_state_dict(state_dict)

        print("Pesos fine-tuned aplicados com sucesso em 'recognizer_system.model'.")
    except Exception as e:
        print(f"ERRO ao carregar state_dict: {e}. O modelo continuará com pesos aleatórios.")
else:
    print(f"ERRO: Arquivo de pesos {parseq_finetuned_path} NÃO ENCONTRADO!")
    print("O modelo 'recognizer_system.model' continuará com pesos NÃO TREINADOS.")

# eval = congela comportamento do modelo, deixa estavel
recognizer_system.eval()
print("Reconhecedor (recognizer_system) pronto para inferência.")

In [ ]:
import os
import json
import cv2
from PIL import Image
from torchvision import transforms
from collections import defaultdict
from google.colab.patches import cv2_imshow
from torchvision import transforms as T
import torch

test_base = '/content/base_competicao/LRLPR-26-5opEvJTW/test'
results_by_category = defaultdict(list)
images_shown_count = 0
max_images_to_show = 3
img_size = recognizer_system.hparams.img_size
parseq_transform = transforms.Compose([
    transforms.Resize(img_size, T.InterpolationMode.BICUBIC),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

print("Iniciando reconhecimento...")
for track_name in sorted(os.listdir(test_base)):
    track_path = os.path.join(test_base, track_name)
    if not os.path.isdir(track_path):
        continue

    annotation_path = os.path.join(track_path, 'annotations.json')
    if not os.path.exists(annotation_path):
        print(f"  - Aviso: sem anotação em {track_name}, pulando.")
        continue

    with open(annotation_path, 'r') as f:
        ann = json.load(f)
    gt_plate = ann.get('plate_text', '').upper()
    category = ann.get('plate_layout', 'unknown')
    if not gt_plate:
        continue

    frames = sorted([f for f in os.listdir(track_path)
                     if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
    if not frames:
        continue

    # acumula predições e scores de todos os frames da track
    track_predictions = []  # lista de (texto, tensor_de_scores_por_char)

    for img_name in frames:
        img_path = os.path.join(track_path, img_name)
        image = cv2.imread(img_path)
        if image is None:
            continue

        if images_shown_count < max_images_to_show:
            print(f"  -> Exibindo: {track_name}/{img_name}")
            cv2_imshow(image)
            images_shown_count += 1

        plate_pil = Image.fromarray(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
        plate_tensor = parseq_transform(plate_pil).unsqueeze(0).to(device)

        with torch.no_grad():
            logits = recognizer_system(images=plate_tensor)

        # desempacota texto e scores por caractere
        texts, scores = recognizer_system.tokenizer.decode(logits.cpu())
        track_predictions.append((texts[0], scores[0]))

    # salva todas as predições da track para aplicar MVCP depois
    results_by_category[category].append({
        'ground_truth': gt_plate,
        'track': track_name,
        'predictions': track_predictions  # lista de (texto, scores)
    })

print("\nReconhecimento concluído.")

Análise de resultado (Acurácia MVCP + CER)

In [ ]:
from collections import defaultdict, Counter
import numpy as np

if "historico_modelos" not in globals():
    historico_modelos = {}

def mvcp(predictions):
    """
    predictions: lista de (texto, tensor_de_scores_por_char)
    Retorna a predição final da track via Majority Vote by Character Position.
    """
    if not predictions:
        return ""

    # limpa caracteres não alfanuméricos (ex: '-' em placas BR)
    clean = []
    for text, scores in predictions:
        text_clean = []
        scores_clean = []
        for char, score in zip(text, scores):
            if char.isalnum():
                text_clean.append(char)
                scores_clean.append(score)
        clean.append(("".join(text_clean), scores_clean))

    # descobre o comprimento mais comum entre as predições
    length = Counter(len(t) for t, _ in clean).most_common(1)[0][0]

    # filtra só predições com esse comprimento
    clean = [(t, s) for t, s in clean if len(t) == length]

    final_chars = []
    for pos in range(length):
        # coleta (caractere, confiança) de cada frame nessa posição
        char_scores = []
        for text, scores in clean:
            char = text[pos]
            # scores é um tensor de shape (seq_len,) com a confiança por posição
            score = scores[pos].item() if pos < len(scores) else 0.0
            char_scores.append((char, score))

        # conta votos por caractere
        vote_count = Counter(c for c, _ in char_scores)
        max_votes = max(vote_count.values())
        candidates = [c for c, v in vote_count.items() if v == max_votes]

        if len(candidates) == 1:
            final_chars.append(candidates[0])
        else:
            # desempate: média de confiança entre os empatados
            best_char = max(
                candidates,
                key=lambda c: sum(s for ch, s in char_scores if ch == c) / vote_count[c]
            )
            final_chars.append(best_char)

    return "".join(final_chars)


def cer(gt, pred):
    """CER via distância de Levenshtein normalizada (igual ao experimento retificado vs não retificado)."""
    n, m = len(gt), len(pred)
    dp = list(range(m + 1))
    for i in range(1, n + 1):
        prev = dp[:]
        dp[0] = i
        for j in range(1, m + 1):
            cost = 0 if gt[i-1] == pred[j-1] else 1
            dp[j] = min(dp[j] + 1, dp[j-1] + 1, prev[j-1] + cost)
    return dp[m] / max(n, 1)


# contagem de acertos, totais e CER por categoria
correct_counts = defaultdict(int)
total_counts = defaultdict(int)
cer_by_category = defaultdict(list)
all_cer_vals = []

for category, results in results_by_category.items():
    if not results:
        continue
    for res in results:
        total_counts[category] += 1
        prediction = mvcp(res['predictions'])
        gt = res['ground_truth']

        if prediction == gt:
            correct_counts[category] += 1

        track_cer = cer(gt, prediction)
        cer_by_category[category].append(track_cer)
        all_cer_vals.append(track_cer)

# resultados de Acurácia (Recognition Rate) e CER
print("\n--- Recognition Rate e CER por Track (MVCP) ---")
total_correct_global = sum(correct_counts.values())
total_plates_global = sum(total_counts.values())

cat_metrics = {}
for category in sorted(total_counts.keys()):
    rr = (correct_counts[category] / total_counts[category]) * 100 if total_counts[category] > 0 else 0.0
    mean_cat_cer = np.mean(cer_by_category[category]) * 100 if cer_by_category[category] else 0.0
    cat_metrics[category] = {"acc": rr, "cer": mean_cat_cer}
    print(f"- Categoria '{category}': Acc = {rr:.2f}% ({correct_counts[category]}/{total_counts[category]} tracks corretas) | CER médio = {mean_cat_cer:.2f}%")

global_rr = (total_correct_global / total_plates_global) * 100 if total_plates_global > 0 else 0.0
global_cer = np.mean(all_cer_vals) * 100 if all_cer_vals else 0.0

print("------------------------------------------")
print(f"Recognition Rate GLOBAL: {global_rr:.2f}% ({total_correct_global}/{total_plates_global} tracks corretas)")
print(f"CER médio GLOBAL:        {global_cer:.2f}%")
print(f"CER = 0 (exatas):        {sum(v == 0 for v in all_cer_vals)} tracks")
print(f"CER > 50%:               {sum(v > 0.5 for v in all_cer_vals)} tracks")
print("------------------------------------------")

historico_modelos["Fine tuning de parseq tiny com base de competição mista cycle gan turbo 25000 passos NAO retificados"] = {
    "Brazilian_acc": cat_metrics.get("Brazilian", {}).get("acc", 0.0),
    "Brazilian_cer": cat_metrics.get("Brazilian", {}).get("cer", 0.0),
    "Mercosur_acc": cat_metrics.get("Mercosur", {}).get("acc", 0.0),
    "Mercosur_cer": cat_metrics.get("Mercosur", {}).get("cer", 0.0),
    "Global_acc": global_rr,
    "Global_cer": global_cer,
}

# Fine tuning de parseq tiny com base de competição mista cycle gan turbo 25000 passos NAO retificado e com TEMPLATES

Carrega tudo e testa

In [ ]:
# carrega modelos
import torch
import os

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Dispositivo de execução: {device.upper()}")

# caminho para pesos salvos no fine-tuning
parseq_finetuned_path = "/content/drive/MyDrive/<CAMINHO_PESOS>/parseq_tiny_finetuned_base_competicao_mista_cyle_rodosol_n_retificado_templates_final.pt"

print("Carregando o sistema ParSeq ...")
recognizer_system = torch.hub.load('baudm/parseq', 'parseq_tiny', pretrained=False).to(device)

# carregar pesos fine-tuned dentro do sub-modelo
print(f"Carregando pesos fine-tuned de: {parseq_finetuned_path}")
if os.path.exists(parseq_finetuned_path):
    try:
        # carrega o state_dict (os pesos)
        state_dict = torch.load(parseq_finetuned_path, map_location=device)

        recognizer_system.model.load_state_dict(state_dict)

        print("Pesos fine-tuned aplicados com sucesso em 'recognizer_system.model'.")
    except Exception as e:
        print(f"ERRO ao carregar state_dict: {e}. O modelo continuará com pesos aleatórios.")
else:
    print(f"ERRO: Arquivo de pesos {parseq_finetuned_path} NÃO ENCONTRADO!")
    print("O modelo 'recognizer_system.model' continuará com pesos NÃO TREINADOS.")

# eval = congela comportamento do modelo, deixa estavel
recognizer_system.eval()
print("Reconhecedor (recognizer_system) pronto para inferência.")

In [ ]:
import os
import json
import cv2
from PIL import Image
from torchvision import transforms
from collections import defaultdict
from google.colab.patches import cv2_imshow
from torchvision import transforms as T
import torch

test_base = '/content/base_competicao/LRLPR-26-5opEvJTW/test'
results_by_category = defaultdict(list)
images_shown_count = 0
max_images_to_show = 3
img_size = recognizer_system.hparams.img_size
parseq_transform = transforms.Compose([
    transforms.Resize(img_size, T.InterpolationMode.BICUBIC),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

print("Iniciando reconhecimento...")
for track_name in sorted(os.listdir(test_base)):
    track_path = os.path.join(test_base, track_name)
    if not os.path.isdir(track_path):
        continue

    annotation_path = os.path.join(track_path, 'annotations.json')
    if not os.path.exists(annotation_path):
        print(f"  - Aviso: sem anotação em {track_name}, pulando.")
        continue

    with open(annotation_path, 'r') as f:
        ann = json.load(f)
    gt_plate = ann.get('plate_text', '').upper()
    category = ann.get('plate_layout', 'unknown')
    if not gt_plate:
        continue

    frames = sorted([f for f in os.listdir(track_path)
                     if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
    if not frames:
        continue

    # acumula predições e scores de todos os frames da track
    track_predictions = []  # lista de (texto, tensor_de_scores_por_char)

    for img_name in frames:
        img_path = os.path.join(track_path, img_name)
        image = cv2.imread(img_path)
        if image is None:
            continue

        if images_shown_count < max_images_to_show:
            print(f"  -> Exibindo: {track_name}/{img_name}")
            cv2_imshow(image)
            images_shown_count += 1

        plate_pil = Image.fromarray(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
        plate_tensor = parseq_transform(plate_pil).unsqueeze(0).to(device)

        with torch.no_grad():
            logits = recognizer_system(images=plate_tensor)

        # desempacota texto e scores por caractere
        texts, scores = recognizer_system.tokenizer.decode(logits.cpu())
        track_predictions.append((texts[0], scores[0]))

    # salva todas as predições da track para aplicar MVCP depois
    results_by_category[category].append({
        'ground_truth': gt_plate,
        'track': track_name,
        'predictions': track_predictions  # lista de (texto, scores)
    })

print("\nReconhecimento concluído.")

Análise de resultado (Acurácia MVCP + CER)

In [ ]:
from collections import defaultdict, Counter
import numpy as np

if "historico_modelos" not in globals():
    historico_modelos = {}

def mvcp(predictions):
    """
    predictions: lista de (texto, tensor_de_scores_por_char)
    Retorna a predição final da track via Majority Vote by Character Position.
    """
    if not predictions:
        return ""

    # limpa caracteres não alfanuméricos (ex: '-' em placas BR)
    clean = []
    for text, scores in predictions:
        text_clean = []
        scores_clean = []
        for char, score in zip(text, scores):
            if char.isalnum():
                text_clean.append(char)
                scores_clean.append(score)
        clean.append(("".join(text_clean), scores_clean))

    # descobre o comprimento mais comum entre as predições
    length = Counter(len(t) for t, _ in clean).most_common(1)[0][0]

    # filtra só predições com esse comprimento
    clean = [(t, s) for t, s in clean if len(t) == length]

    final_chars = []
    for pos in range(length):
        # coleta (caractere, confiança) de cada frame nessa posição
        char_scores = []
        for text, scores in clean:
            char = text[pos]
            # scores é um tensor de shape (seq_len,) com a confiança por posição
            score = scores[pos].item() if pos < len(scores) else 0.0
            char_scores.append((char, score))

        # conta votos por caractere
        vote_count = Counter(c for c, _ in char_scores)
        max_votes = max(vote_count.values())
        candidates = [c for c, v in vote_count.items() if v == max_votes]

        if len(candidates) == 1:
            final_chars.append(candidates[0])
        else:
            # desempate: média de confiança entre os empatados
            best_char = max(
                candidates,
                key=lambda c: sum(s for ch, s in char_scores if ch == c) / vote_count[c]
            )
            final_chars.append(best_char)

    return "".join(final_chars)


def cer(gt, pred):
    """CER via distância de Levenshtein normalizada (igual ao experimento retificado vs não retificado)."""
    n, m = len(gt), len(pred)
    dp = list(range(m + 1))
    for i in range(1, n + 1):
        prev = dp[:]
        dp[0] = i
        for j in range(1, m + 1):
            cost = 0 if gt[i-1] == pred[j-1] else 1
            dp[j] = min(dp[j] + 1, dp[j-1] + 1, prev[j-1] + cost)
    return dp[m] / max(n, 1)


# contagem de acertos, totais e CER por categoria
correct_counts = defaultdict(int)
total_counts = defaultdict(int)
cer_by_category = defaultdict(list)
all_cer_vals = []

for category, results in results_by_category.items():
    if not results:
        continue
    for res in results:
        total_counts[category] += 1
        prediction = mvcp(res['predictions'])
        gt = res['ground_truth']

        if prediction == gt:
            correct_counts[category] += 1

        track_cer = cer(gt, prediction)
        cer_by_category[category].append(track_cer)
        all_cer_vals.append(track_cer)

# resultados de Acurácia (Recognition Rate) e CER
print("\n--- Recognition Rate e CER por Track (MVCP) ---")
total_correct_global = sum(correct_counts.values())
total_plates_global = sum(total_counts.values())

cat_metrics = {}
for category in sorted(total_counts.keys()):
    rr = (correct_counts[category] / total_counts[category]) * 100 if total_counts[category] > 0 else 0.0
    mean_cat_cer = np.mean(cer_by_category[category]) * 100 if cer_by_category[category] else 0.0
    cat_metrics[category] = {"acc": rr, "cer": mean_cat_cer}
    print(f"- Categoria '{category}': Acc = {rr:.2f}% ({correct_counts[category]}/{total_counts[category]} tracks corretas) | CER médio = {mean_cat_cer:.2f}%")

global_rr = (total_correct_global / total_plates_global) * 100 if total_plates_global > 0 else 0.0
global_cer = np.mean(all_cer_vals) * 100 if all_cer_vals else 0.0

print("------------------------------------------")
print(f"Recognition Rate GLOBAL: {global_rr:.2f}% ({total_correct_global}/{total_plates_global} tracks corretas)")
print(f"CER médio GLOBAL:        {global_cer:.2f}%")
print(f"CER = 0 (exatas):        {sum(v == 0 for v in all_cer_vals)} tracks")
print(f"CER > 50%:               {sum(v > 0.5 for v in all_cer_vals)} tracks")
print("------------------------------------------")

historico_modelos["Fine tuning de parseq tiny com base de competição mista cycle gan turbo 25000 passos NAO retificado e com TEMPLATES"] = {
    "Brazilian_acc": cat_metrics.get("Brazilian", {}).get("acc", 0.0),
    "Brazilian_cer": cat_metrics.get("Brazilian", {}).get("cer", 0.0),
    "Mercosur_acc": cat_metrics.get("Mercosur", {}).get("acc", 0.0),
    "Mercosur_cer": cat_metrics.get("Mercosur", {}).get("cer", 0.0),
    "Global_acc": global_rr,
    "Global_cer": global_cer,
}

# Resumo Comparativo (Acurácia e CER de Todos os Modelos)

In [ ]:
import pandas as pd

if historico_modelos:
    df_resumo = pd.DataFrame.from_dict(historico_modelos, orient='index')
    df_resumo.index.name = 'Modelo'
    df_resumo = df_resumo.sort_values(by='Global_acc', ascending=False)
    print("\n" + "="*105)
    print(f"{'RESUMO GERAL DOS MODELOS (MVCP por Track)':^105}")
    print("="*105)
    print(df_resumo.to_string(float_format=lambda x: f"{x:.2f}%"))
    print("="*105)
else:
    print("Nenhum modelo foi avaliado ainda na sessão.")